# Cobalt L-edge Energy Sweep Tutorial

This notebook keeps the sample, object-hole aperture, reference holes, illumination, and magnetic pattern fixed, then simulates Fourier transform holography at several photon energies across the Cobalt L3 and L2 edges.

The main diagnostics are:

- `CR` and `CL` exit waves at each energy;
- the complex XMCD exit-wave image `log(exit_CR / exit_CL)` in the object-hole ROI;
- XMCD amplitude and phase spectra from positive/negative magnetic-domain averages;
- ideal and detected holograms;
- FTH reconstructions cropped to the object-hole ROI;
- total hologram intensity as a function of photon energy.

The default grid is deliberately small enough for interactive use. Increase `detector_shape`, `energy_values_eV`, or enable multislice propagation once the workflow is clear.


In [ ]:
import numpy as np
import matplotlib.pyplot as plt

%matplotlib widget
plt.rcParams["figure.constrained_layout.use"] = True

from scattering_calculator.sample_generator import pattern_generator
from scattering_calculator.simulation_pipelines import simulation_configuration as sim


## 1. User Controls

The geometry below is fixed for the whole sweep. Only `energy` changes inside the loop.

`RUN_ENERGY_SWEEP=False` lets you inspect the notebook quickly without starting the simulations. Set it to `True` to generate the energy-dependent arrays.


In [ ]:
RUN_ENERGY_SWEEP = True

# Energies spanning the Co L3 edge near 778 eV and the L2 edge near 793 eV.
energy_values_eV = np.arange(773.0, 803.0, 1.0)
reference_energy_eV = 778.0

# Use Jones for the full polarization interaction. Scalar is faster and can be
# used when the selected polarization is an eigenmode of the local interaction.
propagator_method = "Scalar"
scalar_refractive_index_lazy = True

# Jones-only is fast and already includes the rank-zero longitudinal phase.
# Set propagate=True to include multislice free-space propagation between layers.
propagate = False
jones_apply_zero_order_phase = True
multislice_propagation_roi = True
multislice_propagation_roi_padding_px = 4
multislice_propagation_roi_merge_overlaps = True
propagation_padding_px = 4
propagation_padding_mode = "edge"
propagation_absorber_width_px = 2
propagation_absorber_strength = 6.0
propagation_absorber_profile = "cosine"

# Detector and beam settings. The sample mesh is fixed from reference_energy_eV.
detector_shape = (512, 512)
detector_center = tuple(np.array(detector_shape) // 2)
oversampling = 1
photon_flux = 5e8
coherence_length = (250e-6, 250e-6)

# Fixed sample, aperture, illumination, and magnetic texture.
recipe = "[Au(300)/Cr(30)]x5/SiN(80)/Pt(10)Co(15)"
sample_name = "cobalt_l_edge_energy_sweep_tutorial"
illumination_center = (0.0, 0.0)
illumination_focus_distance = 1e-3
illumination_fwhm = 50.e-6
illumination_alpha_beam = (0.0, 0.0)

aperture_types = ["OH", "RH", "RH"]
aperture_radii = [555e-9, 60e-9, 60e-9]
aperture_centers = [(0.0, 0.0), (-2040e-9, -2025e-9), (2025e-9, -2000e-9)]
aperture_sigmas = [4e-9, 2e-9, 2e-9]
aperture_angles = [0.0, 0.0, 0.0]
aperture_ellipticities = [1.0, 1.0, 1.0]
aperture_roughnesses = [0.0, 0.02, 0.02]
aperture_roughness_modes = [(0, 0), (3, 10), (3, 10)]
aperture_seeds = [1, 2, 3]
aperture_top_radius_factors = [1.0,3., 3.]

# The object image in an FTH reconstruction is not centered at the origin.
# It is displaced by the reference-hole/object-hole separation. Use the first
# RH by default; flip the sign if you want the conjugate/twin object image.
fth_reference_aperture_index = 1
fth_reconstruction_shift_sign = 1

# Domain averages ignore pixels close to a domain wall, where mz is neither
# clearly positive nor clearly negative after smoothing.
xmcd_domain_threshold = 0.9

pattern_type = "binary_labyrinth_pattern"
pattern_config = {
    "stripe_width": 75e-9,
    "sigma": 4e-9,
    "domain_conversion": "soft",
    "softness": 1.2,
    "n_steps": 60,
    "seed": 4,
    "use_gpu": False,
}


## 2. Build The Fixed Sample Mesh And Magnetic Pattern

The detector geometry and reference energy define a fixed sample-plane pixel size. The same physical magnetic pattern is reused at every energy, so changes in the result come from the energy-dependent refractive indices rather than from a different random sample.


In [ ]:
def make_xray_config(energy_eV, pol="CR"):
    xray_config = sim.XRayConfig(
        energy=float(energy_eV),
        photon_flux=photon_flux,
        pol=pol,
        coherence_length=coherence_length,
    )
    xray_config.setup()
    return xray_config


def make_detector_config(noise_seed=20):
    beamstop_config = sim.BeamstopConfig(
        bs_method="circular",
        bs_detector_distance=0.010,
        bs_center=detector_center,
        bs_config={
            "radius": 450e-6,
            "sigma": 50e-6,
            "wire_width": 40e-6,
            "wire_bend": 30e-6,
            "angle": np.deg2rad(25),
            "antialias": 2,
            "seed": 4,
        },
    )
    detector_config = sim.DetectorConfig(
        shape=detector_shape,
        pixel_size=10e-6,
        sample_to_detector_distance=0.075,
        detector_center=detector_center,
        detector_params={
            "readout_noise_average": 10,
            "readout_noise_sigma": 1,
            "detector_threshold": 64e3,
            "counts_per_photon": 200,
            "quantum_efficiency": 1.0,
            "noise_seed": int(noise_seed),
        },
        measurement_config={
            "number_frames": 200,
            "max_counts_per_image": 64e3,
            "exposure_time": 1.0,
        },
        beamstop_config=beamstop_config,
        use_detector_pixel_footprint=True,
        detector_pixel_footprint_samples=oversampling,
    )
    detector_config.setup()
    return detector_config


reference_xray_config = make_xray_config(reference_energy_eV)
reference_detector_config = make_detector_config()
real_space_pixel_size = reference_detector_config.calc_realspace_resolution(reference_xray_config.beam_params) / oversampling
sample_shape = np.array([0, oversampling * detector_shape[0], oversampling * detector_shape[1]], dtype=int)

magnetic_pattern_config = sim.MagneticPatternConfig(
    pattern_type_method=pattern_type,
    shape=tuple(sample_shape[1:]),
    real_space_pixel_size=real_space_pixel_size,
    pattern_config=pattern_config,
)
magnetic_pattern_config.create_pattern()
magnetic_pattern_2d = magnetic_pattern_config.magnetic_pattern

print(f"Fixed sample pixel size: {real_space_pixel_size * 1e9:.2f} nm")
print(f"Fixed lateral grid: {tuple(sample_shape[1:])}")
print(f"Energy points: {len(energy_values_eV)}")


In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(9, 4))
extent_nm = np.array([-sample_shape[2] / 2, sample_shape[2] / 2, sample_shape[1] / 2, -sample_shape[1] / 2]) * real_space_pixel_size * 1e9
im = axes[0].imshow(magnetic_pattern_2d, cmap="RdBu_r", vmin=-1, vmax=1, extent=extent_nm, interpolation="nearest")
axes[0].set_title("fixed magnetic pattern mz")
axes[0].set_xlabel("x (nm)")
axes[0].set_ylabel("y (nm)")
fig.colorbar(im, ax=axes[0], label="mz")

# Draw the configured aperture centers and nominal bottom radii on top of the pattern.
axes[1].imshow(magnetic_pattern_2d, cmap="gray", extent=extent_nm, interpolation="nearest")
for typ, radius, center in zip(aperture_types, aperture_radii, aperture_centers):
    cy, cx = center
    circle = plt.Circle((cx * 1e9, cy * 1e9), radius * 1e9, fill=False, linewidth=1.5)
    axes[1].add_patch(circle)
    axes[1].plot(cx * 1e9, cy * 1e9, "+", markersize=8)
    axes[1].text(cx * 1e9, cy * 1e9, f" {typ}", va="center")
axes[1].set_title("aperture layout over fixed pattern")
axes[1].set_xlabel("x (nm)")
axes[1].set_ylabel("y (nm)")


## 3. Simulation Helpers

At each energy the notebook rebuilds the material optical constants and the dielectric tensor, because those are energy dependent. The magnetic pattern and aperture parameters are copied unchanged.

The XMCD exit-wave image is defined as:

```python
xmcd_log = log(exit_wave_CR / exit_wave_CL)
```

Its real part is the log-amplitude ratio. Its imaginary part is the phase difference, wrapped by the complex logarithm.


In [ ]:
def frame_average(array):
    array = np.asarray(array)
    return array.mean(axis=0) if array.ndim == 3 else array


def symmetric_limits(data, percentile=99.0):
    vmax = np.nanpercentile(np.abs(data), percentile)
    return -float(vmax or 1.0), float(vmax or 1.0)


def continuous_phase(values):
    """Return a phase curve without artificial 2*pi branch-cut jumps."""
    phase = np.unwrap(np.angle(values))
    offset = 2 * np.pi * np.round(np.nanmedian(phase) / (2 * np.pi))
    return phase - offset


def continuous_complex_log_ratio(numerator, denominator, eps=1e-30):
    """Compute log(numerator / denominator) with an unwrapped imaginary part."""
    ratio = (np.asarray(numerator) + eps) / (np.asarray(denominator) + eps)
    return np.log(np.abs(ratio)) + 1j * continuous_phase(ratio)


def resample_nearest_to_shape(array, shape):
    """Nearest-neighbor resample a 2D mask/image to a requested shape."""
    array = np.asarray(array)
    if array.shape == tuple(shape):
        return array
    y_idx = np.rint(np.linspace(0, array.shape[0] - 1, shape[0])).astype(int)
    x_idx = np.rint(np.linspace(0, array.shape[1] - 1, shape[1])).astype(int)
    return array[np.ix_(y_idx, x_idx)]


def object_hole_crop_slices(shape, pixel_size, radius_factor=1.0):
    oh_index = aperture_types.index("OH")
    center_y_m, center_x_m = aperture_centers[oh_index]
    return crop_slices_around_position(shape, pixel_size, (center_y_m, center_x_m), aperture_radii[oh_index] * radius_factor)


def fth_object_image_crop_slices(shape, pixel_size, radius_factor=1.0):
    """Crop the FTH object image displaced by the selected RH-OH vector.

    In the exit wave the object hole is centered at its real-space OH position.
    In the FTH reconstruction, the object image appears in the cross-correlation
    term, shifted from the reconstruction origin by the reference-hole/object-hole
    separation. The sign selects which twin image to inspect.
    """
    oh_index = aperture_types.index("OH")
    oh_center = np.asarray(aperture_centers[oh_index], dtype=float)
    rh_center = np.asarray(aperture_centers[fth_reference_aperture_index], dtype=float)
    reconstruction_center = fth_reconstruction_shift_sign * (rh_center - oh_center)
    return crop_slices_around_position(
        shape,
        pixel_size,
        reconstruction_center,
        aperture_radii[oh_index] * radius_factor,
    )


def crop_slices_around_position(shape, pixel_size, center_m, radius_m):
    center_y_m, center_x_m = center_m
    cy = int(np.clip(round(shape[-2] / 2 + center_y_m / pixel_size), 0, shape[-2] - 1))
    cx = int(np.clip(round(shape[-1] / 2 + center_x_m / pixel_size), 0, shape[-1] - 1))
    half_width = max(4, int(np.ceil(radius_m / pixel_size)))
    y0 = max(0, cy - half_width)
    y1 = min(shape[-2], cy + half_width + 1)
    x0 = max(0, cx - half_width)
    x1 = min(shape[-1], cx + half_width + 1)
    return slice(y0, y1), slice(x0, x1)


sample_oh_slices = object_hole_crop_slices(sample_shape[1:], real_space_pixel_size, radius_factor=1.0)


def build_sample_and_aperture(xray_config):
    sample_config = sim.SampleConfig(
        recipe=recipe,
        sample_shape=sample_shape.copy(),
        real_space_pixel_size=real_space_pixel_size,
        xray_config=xray_config,
        sample_name=sample_name,
    )
    sample_config.setup()

    magnetization = pattern_generator.map_magnetization_to_3d(
        magnetic_pattern_x=np.zeros_like(magnetic_pattern_2d),
        magnetic_pattern_y=np.sqrt(np.clip(1 - np.abs(magnetic_pattern_2d) ** 2, 0, 1)),
        magnetic_pattern_z=magnetic_pattern_2d,
        nr_repeats=sample_config.sample_structure.sample_shape[0],
    )
    sample_config.assign_magnetic_pattern(magnetization)

    layer_thicknesses = sample_config.sample_structure.layer_thicknesses
    membrane_index = sample_config.sample_structure.layer_names.index("SiN")
    aperture_taper_depth = float(np.sum(layer_thicknesses[: max(0, membrane_index - 2)]))
    thickness_oh = float(np.sum(layer_thicknesses[:membrane_index]))

    aperture_config = sim.FrontApertureConfig(
        aperture_method="FTH_circular",
        aperture_shape=sample_config.sample_structure.sample_shape,
        real_space_pixel_size=real_space_pixel_size,
        aperture_thicknesses=layer_thicknesses,
        use_roi=True,
        aperture_config={
            "apertures_type": aperture_types,
            "apertures_radius": aperture_radii,
            "apertures_center": aperture_centers,
            "apertures_sigma": aperture_sigmas,
            "apertures_angle": aperture_angles,
            "apertures_ellipticity": aperture_ellipticities,
            "apertures_roughness": aperture_roughnesses,
            "apertures_roughness_modes": aperture_roughness_modes,
            "apertures_seed": aperture_seeds,
            "apertures_top_radius_factor": aperture_top_radius_factors,
            "aperture_taper_depth": aperture_taper_depth,
            "thickness_OH": thickness_oh,
        },
    )
    aperture_config.setup()
    aperture_mask = aperture_config.return_aperture()
    sample_config.assign_aperture_mask(aperture_mask)

    if propagator_method == "Scalar":
        sample_config.sample_structure.calculate_final_scalar_refractive_index(
            pol="CR",
            use_aperture_roi=True,
            compact=True,
            lazy=scalar_refractive_index_lazy,
        )
    else:
        sample_config.sample_structure.calculate_final_dielectric_tensor(
            use_aperture_roi=True,
            compact=True,
        )
    return sample_config, aperture_config, aperture_mask


def simulate_one_energy(energy_eV, noise_seed_base=1000):
    xray_config = make_xray_config(energy_eV)
    detector_config = make_detector_config(noise_seed=noise_seed_base)
    detector_config.calc_realspace_resolution(xray_config.beam_params)
    sample_config, aperture_config, aperture_mask = build_sample_and_aperture(xray_config)

    illumination_config = sim.IlluminationConfig(
        XRayConfig=xray_config,
        shape=tuple(sample_config.sample_structure.sample_shape[1:]),
        real_space_pixel_size=real_space_pixel_size,
        illumination_function="gaussian",
        illumination_config={
            "center": np.array(illumination_center),
            "distance": illumination_focus_distance,
            "fwhm": illumination_fwhm,
            "alpha_beam": illumination_alpha_beam,
        },
    )
    illumination_config.setup()

    hologram_config = sim.HologramConfig(
        sample_x=sample_config.sample_structure.x,
        sample_y=sample_config.sample_structure.y,
        detector_layout=detector_config.detector_layout,
    )

    for pol_index, pol in enumerate(["CR", "CL"]):
        detector_config.detector_params["noise_seed"] = int(noise_seed_base) + pol_index
        illumination_config.update_polarization(pol)
        propagator_config = sim.SamplePropagatorConfig(
            SampleConfig=sample_config,
            IlluminationConfig=illumination_config,
            propagator_method=propagator_method,
            propagator_config={
                "propagator_method": propagator_method,
                "propagate": propagate,
                "jones_apply_zero_order_phase": jones_apply_zero_order_phase,
                "scalar_apply_zero_order_phase": jones_apply_zero_order_phase,
                "scalar_refractive_index_lazy": scalar_refractive_index_lazy,
                "propagation_padding_px": propagation_padding_px,
                "propagation_padding_mode": propagation_padding_mode,
                "propagation_absorber_width_px": propagation_absorber_width_px,
                "propagation_absorber_strength": propagation_absorber_strength,
                "propagation_absorber_profile": propagation_absorber_profile,
                "multislice_propagation_roi": multislice_propagation_roi,
                "multislice_propagation_roi_padding_px": multislice_propagation_roi_padding_px,
                "multislice_propagation_roi_merge_overlaps": multislice_propagation_roi_merge_overlaps,
            },
        )
        propagator_config.setup()
        detector_config.assign_propagated_wavefront(propagator_config)
        detector_config.detect_hologram()

        hologram_config.add_exit_waves({pol: propagator_config.return_scalar_wavefield()})
        hologram_config.add_holograms({pol: detector_config.return_ideal_hologram()}, source="ideal")
        hologram_config.add_holograms({pol: detector_config.return_detected_hologram(store_no_beamstop=True)}, source="detected")
        hologram_config.add_holograms({pol: detector_config.return_detected_hologram_without_beamstop()}, source="detected_no_beamstop")

    hologram_config.compute_differences()
    hologram_config.compute_sums()
    hologram_config.compute_reconstructions()
    hologram_config.compute_averages()

    eps = 1e-30
    xmcd_ratio = (hologram_config.exit_waves["CR"] + eps) / (hologram_config.exit_waves["CL"] + eps)
    xmcd_log = np.log(xmcd_ratio)
    reconstruction_pixel_size = detector_config.detector_layout.real_space_resolution
    recon_oh_slices = fth_object_image_crop_slices(detector_shape, reconstruction_pixel_size, radius_factor=1.0)
    oh_index = aperture_types.index("OH")
    fth_shift_m = fth_reconstruction_shift_sign * (
        np.asarray(aperture_centers[fth_reference_aperture_index], dtype=float)
        - np.asarray(aperture_centers[oh_index], dtype=float)
    )

    return {
        "energy_eV": float(energy_eV),
        "xray_config": xray_config,
        "detector_config": detector_config,
        "sample_config": sample_config,
        "aperture_config": aperture_config,
        "aperture_mask": aperture_mask,
        "holograms": hologram_config,
        "xmcd_ratio": xmcd_ratio,
        "xmcd_log": xmcd_log,
        "sample_oh_slices": sample_oh_slices,
        "recon_oh_slices": recon_oh_slices,
        "fth_reconstruction_shift_m": fth_shift_m,
        "total_ideal_intensity": float(np.sum(hologram_config.ideal_holograms["sum"])),
        "total_detected_counts": float(np.sum(hologram_config.detected_holograms["sum"])),
    }


## 4. Preview The Fixed Aperture Mask

This cell uses the reference energy only to build the material stack. The aperture geometry is then reused at every energy.


In [ ]:
preview_sample, preview_aperture, preview_aperture_mask = build_sample_and_aperture(reference_xray_config)
projection = np.mean(preview_aperture_mask, axis=0)

fig, axes = plt.subplots(1, 2, figsize=(9, 4))
im0 = axes[0].imshow(projection, cmap="gray", vmin=0, vmax=1, extent=extent_nm, interpolation="nearest")
axes[0].set_title("front aperture projection")
axes[0].set_xlabel("x (nm)")
axes[0].set_ylabel("y (nm)")
fig.colorbar(im0, ax=axes[0], label="material fraction")

masked_pattern = magnetic_pattern_2d * (projection < 0.5)
im1 = axes[1].imshow(masked_pattern, cmap="RdBu_r", vmin=-1, vmax=1, extent=extent_nm, interpolation="nearest")
axes[1].set_title("magnetic pattern visible through apertures")
axes[1].set_xlabel("x (nm)")
axes[1].set_ylabel("y (nm)")
fig.colorbar(im1, ax=axes[1], label="mz")


## 5. Run The Energy Sweep

The loop stores a small dictionary per energy. The `holograms` entry is a `HologramConfig` containing `CR`, `CL`, `diff = CR - CL`, `sum = CR + CL`, and FTH reconstructions for each stored channel.


In [ ]:
energy_results = []

if RUN_ENERGY_SWEEP:
    for i, energy in enumerate(energy_values_eV):
        print(f"[{i + 1:02d}/{len(energy_values_eV)}] Simulating {energy:.2f} eV")
        energy_results.append(simulate_one_energy(energy, noise_seed_base=1000 + 10 * i))
    print("Done.")
else:
    print("Sweep disabled. Set RUN_ENERGY_SWEEP = True and rerun this cell.")


## 6. XMCD Exit-Wave Log Ratio

The complex image below is evaluated inside the object-hole ROI:

- `real(log(CR/CL)) = log(abs(CR/CL))`: signed absorption XMCD amplitude;
- `imag(log(CR/CL))`: signed phase XMCD in radians;
- `abs(CR/CL)`: raw helicity-ratio modulus, useful for intuition;
- `angle(CR/CL)`: raw helicity-ratio phase before taking the complex log.

Each column uses one fixed colorbar range across all displayed energies, so apparent contrast changes are not caused by automatic rescaling.


### XMCD Amplitude And Phase Definitions

Following the ratio-contrast convention, the complex XMCD signal is defined as

```python
XMCD = log(exit_wave_CR / exit_wave_CL)
```

The two physically useful signed quantities are then

- `$A_\mathrm{XMCD} = \log{|\phi_\mathrm{CR}/\phi_\mathrm{CL}|} = \mathrm{Re}[\log(\phi_\mathrm{CR}/\phi_\mathrm{CL})]$`: absorption XMCD amplitude;
- `$\Theta_\mathrm{XMCD} = \arg(\phi_\mathrm{CR}/\phi_\mathrm{CL}) = \mathrm{Im}[\log(\phi_\mathrm{CR}/\phi_\mathrm{CL})]$`: phase XMCD.

Both are signed. Opposite magnetic domains should therefore appear with opposite signs in `$A_\mathrm{XMCD}$` and `$\Theta_\mathrm{XMCD}$`. The quantity `abs(log(CR/CL))` is not used as the XMCD amplitude because it removes that sign.


In [ ]:
if energy_results:
    selected_indices = sorted(set(np.arange(0,len(energy_results),16)))
    rows = []
    for idx in selected_indices:
        result = energy_results[idx]
        crop = result["sample_oh_slices"]
        data = frame_average(result["xmcd_log"])[crop]
        ratio = frame_average(result["xmcd_ratio"])[crop]
        rows.append(
            {
                "energy_eV": result["energy_eV"],
                "panels": [
                    (np.real(data), "$A_{\\mathrm{XMCD}}$", "RdBu_r", True),
                    (np.imag(data), "$\\Theta_{\\mathrm{XMCD}}$", "twilight", True),
                    (np.abs(ratio), "abs(CR/CL)", "magma", False),
                    (np.angle(ratio), "angle(CR/CL)", "twilight", True),
                ],
            }
        )

    column_limits = []
    for col in range(4):
        images = [row["panels"][col][0] for row in rows]
        symmetric = rows[0]["panels"][col][3]
        stacked = np.concatenate([np.ravel(image) for image in images])
        if symmetric:
            vmax = np.nanpercentile(np.abs(stacked), 99)
            column_limits.append((-float(vmax or 1.0), float(vmax or 1.0)))
        else:
            column_limits.append(tuple(np.nanpercentile(stacked, [1, 99])))

    fig, axes = plt.subplots(len(rows), 4, figsize=(14, 3.2 * len(rows)), squeeze=False)

    for row_index, row in enumerate(rows):
        panels = row["panels"]
        for col, (image, title, cmap, symmetric) in enumerate(panels):
            ax = axes[row_index, col]
            vmin, vmax = column_limits[col]
            im = ax.imshow(image, cmap=cmap, vmin=vmin, vmax=vmax, interpolation="nearest")
            ax.set_title(f"{row['energy_eV']:.1f} eV: {title}")
            ax.set_xlabel("x in OH ROI (px)")
            ax.set_ylabel("y in OH ROI (px)")
            fig.colorbar(im, ax=ax, shrink=0.8)
else:
    print("No sweep results available yet.")


## 7. Holograms Across The Edge

The detected `CR`, `CL`, and `CR - CL` holograms are shown for a few representative energies. The individual helicity holograms are plotted on a logarithmic scale because the central intensity dominates the dynamic range.

Colorbar limits are fixed per column across all displayed energies.


In [ ]:
if energy_results:
    selected_indices = sorted(set(np.arange(0,len(energy_results),8)))
    rows = []
    for idx in selected_indices:
        result = energy_results[idx]
        h = result["holograms"]
        cr = frame_average(h.ideal_holograms["CR"])
        cl = frame_average(h.detected_holograms["CL"])
        diff = frame_average(h.detected_holograms["diff"])
        rows.append(
            {
                "energy_eV": result["energy_eV"],
                "panels": [
                    (np.log10(np.maximum(cr, 1.0)), "log10 ideal CR", "viridis", False),
                    (np.log10(np.maximum(cl, 1.0)), "log10 detected CL", "viridis", False),
                    (diff, "detected CR - CL", "RdBu_r", True),
                ],
            }
        )

    column_limits = []
    for col in range(3):
        images = [row["panels"][col][0] for row in rows]
        symmetric = rows[0]["panels"][col][3]
        stacked = np.concatenate([np.ravel(image) for image in images])
        if symmetric:
            vmax = np.nanpercentile(np.abs(stacked), 99)
            column_limits.append((-float(vmax or 1.0), float(vmax or 1.0)))
        else:
            column_limits.append(tuple(np.nanpercentile(stacked, [1, 99])))

    fig, axes = plt.subplots(len(rows), 3, figsize=(11, 3.5 * len(rows)), squeeze=False)

    for row_index, row in enumerate(rows):
        panels = row["panels"]
        for col, (image, title, cmap, symmetric) in enumerate(panels):
            ax = axes[row_index, col]
            vmin, vmax = column_limits[col]
            im = ax.imshow(image, cmap=cmap, vmin=vmin, vmax=vmax, interpolation="nearest")
            ax.set_title(f"{row['energy_eV']:.1f} eV: {title}")
            ax.set_xlabel("detector x (px)")
            ax.set_ylabel("detector y (px)")
            fig.colorbar(im, ax=ax, shrink=0.8)
else:
    print("No sweep results available yet.")


## 8. FTH Reconstructions Cropped To The Shifted Object Image

The object-hole crop used for the exit wave is centered on the real-space OH. The FTH reconstruction is different: the reconstructed object appears in the OH/RH cross-correlation term, shifted away from the reconstruction origin by the reference-hole/object-hole separation vector.

This notebook uses `fth_reference_aperture_index` to pick the RH and crops around `fth_reconstruction_shift_sign * (RH_center - OH_center)`. Flip `fth_reconstruction_shift_sign` if you want the conjugate/twin object image on the opposite side.

The displayed crop is intentionally tight: it uses the physical OH radius without the larger visualization padding used in some broader overview notebooks. Amplitude colorbar limits are fixed per column across all displayed energies. Phase is always shown on `[-pi, pi]`.


In [ ]:
if energy_results:
    selected_indices = sorted(set(np.arange(0,len(energy_results),8)))
    rows = []
    for idx in selected_indices:
        result = energy_results[idx]
        h = result["holograms"]
        crop = result["recon_oh_slices"]
        ideal = frame_average(h.reconstructions["ideal"]["diff"])[crop]
        detected = frame_average(h.reconstructions["detected"]["diff"])[crop]
        rows.append(
            {
                "energy_eV": result["energy_eV"],
                "panels": [
                    (np.real(ideal), "ideal Re(CR-CL)", "gray", False),
                    (np.imag(ideal), "ideal Im(CR-CL)", "gray", False),
                    (np.real(detected), "detected Re(CR-CL)", "gray", False),
                    (np.imag(detected), "detected Im(CR-CL)", "gray", False),
                ],
            }
        )

    column_limits = []
    for col in range(4):
        images = [row["panels"][col][0] for row in rows]
        symmetric = rows[0]["panels"][col][3]
        if symmetric:
            column_limits.append((-np.pi, np.pi))
        else:
            stacked = np.concatenate([np.ravel(image) for image in images])
            column_limits.append(tuple(np.nanpercentile(stacked, [1, 99])))

    fig, axes = plt.subplots(len(rows), 4, figsize=(14, 3.2 * len(rows)), squeeze=False)

    for row_index, row in enumerate(rows):
        panels = row["panels"]
        for col, (image, title, cmap, symmetric) in enumerate(panels):
            ax = axes[row_index, col]
            vmin, vmax = column_limits[col]
            im = ax.imshow(image, cmap=cmap, vmin=vmin, vmax=vmax, interpolation="nearest")
            ax.set_title(f"{row['energy_eV']:.1f} eV: {title}")
            ax.set_xlabel("x in FTH ROI (px)")
            ax.set_ylabel("y in FTH ROI (px)")
            fig.colorbar(im, ax=ax, shrink=0.8)
else:
    print("No sweep results available yet.")


## 9. Ideal FTH XMCD Images In The Shifted Object Crop

The panels below show the XMCD ratio extracted directly from the ideal FTH reconstructions:

```python
FTH_XMCD = log(FTH_CR / FTH_CL)
```

The plotted maps are `$A_\mathrm{XMCD} = \mathrm{Re}[\log(FTH_CR/FTH_CL)]$` and `$\Theta_\mathrm{XMCD} = \mathrm{Im}[\log(FTH_CR/FTH_CL)]$`. The crop is the shifted FTH object image crop, centered at `fth_reconstruction_shift_sign * (RH_center - OH_center)`, not at the reconstruction origin.

Colorbar limits are fixed per column across all displayed energies.


In [ ]:
if energy_results:
    selected_indices = sorted(set(np.arange(0,len(energy_results),8)))
    rows = []
    eps = 1e-30
    for idx in selected_indices:
        result = energy_results[idx]
        h = result["holograms"]
        crop = result["recon_oh_slices"]
        fth_cr = frame_average(h.reconstructions["ideal"]["CR"])[crop]
        fth_cl = frame_average(h.reconstructions["ideal"]["CL"])[crop]
        fth_ratio = (fth_cr + eps) / (fth_cl + eps)
        fth_xmcd_log = np.log(fth_ratio)
        rows.append(
            {
                "energy_eV": result["energy_eV"],
                "panels": [
                    (np.real(fth_xmcd_log), "$A_{\\mathrm{XMCD}}$ from ideal FTH", "RdBu_r", True),
                    (np.imag(fth_xmcd_log), "$\\Theta_{\\mathrm{XMCD}}$ from ideal FTH", "twilight", True),
                ],
            }
        )

    column_limits = []
    for col in range(2):
        images = [row["panels"][col][0] for row in rows]
        symmetric = rows[0]["panels"][col][3]
        stacked = np.concatenate([np.ravel(image) for image in images])
        if symmetric:
            vmax = np.nanpercentile(np.abs(stacked), 99)
            column_limits.append((-float(vmax or 1.0), float(vmax or 1.0)))
        else:
            column_limits.append(tuple(np.nanpercentile(stacked, [1, 99])))

    fig, axes = plt.subplots(len(rows), 2, figsize=(8, 3.2 * len(rows)), squeeze=False)

    for row_index, row in enumerate(rows):
        panels = row["panels"]
        for col, (image, title, cmap, symmetric) in enumerate(panels):
            ax = axes[row_index, col]
            vmin, vmax = column_limits[col]
            im = ax.imshow(image, cmap=cmap, vmin=vmin, vmax=vmax, interpolation="nearest")
            ax.set_title(f"{row['energy_eV']:.1f} eV: {title}")
            ax.set_xlabel("x in shifted FTH OH ROI (px)")
            ax.set_ylabel("y in shifted FTH OH ROI (px)")
            fig.colorbar(im, ax=ax, shrink=0.8)
else:
    print("No sweep results available yet.")


## 10. XMCD Signal Versus Energy

The image panels show where the XMCD contrast lives. This cell turns that into a spectrum by averaging the complex XMCD signal inside magnetic domains:

```python
XMCD = log(exit_wave_CR / exit_wave_CL)
```

Pixels with `mz > xmcd_domain_threshold` define the positive domain; pixels with `mz < -xmcd_domain_threshold` define the negative domain. Pixels near domain walls are skipped so the average is not dominated by smoothed boundaries.

The plotted complex averages are computed from the domain-averaged complex ratio `exit_wave_CR / exit_wave_CL`. This avoids averaging already-logged pixels before taking another logarithm.

For a pure circular XMCD contribution, reversing `mz` swaps the CR and CL effective indices. That means `log(CR/CL)` changes sign between up and down domains. The signed amplitude contrast is therefore `real(log(CR/CL)) = log(abs(CR/CL))`, not `abs(log(CR/CL))`; the latter discards the sign and can make opposite domains look artificially identical.

- positive-domain `log(mean(CR/CL))`;
- negative-domain `log(mean(CR/CL))`;
- domain contrast, `log(mean(CR/CL)_positive / mean(CR/CL)_negative)`.

The positive/negative domain means are plotted on the left y-axis. The logarithmic domain contrast is plotted on a separate right y-axis, because it can naturally have a different scale.

In the first row, the y values are `$A_\mathrm{XMCD}$`: the signed log-amplitude ratio, and `$\Theta_\mathrm{XMCD}$`: the signed phase of the same helicity ratio, measured directly from the exit waves.

For readability, the exit-wave and ideal-FTH XMCD rows are plotted with vertical offsets: the positive/negative domain means are placed in the upper band, while the green positive-versus-negative XMCD contrast is placed in the lower band. The translucent green fill shows the area between each contrast curve and its own zero baseline.

The second row repeats the ratio analysis on the ideal FTH reconstructions, using `log(reconstruction_CR / reconstruction_CL)` inside the shifted OH image crop. That crop is displaced from the reconstruction center by the selected RH-OH vector, so it follows the actual FTH object image rather than the origin. The plotted quantities are `$A_\mathrm{XMCD}$` and `$\Theta_\mathrm{XMCD}$` extracted from the ideal FTH reconstructions, with the same positive/negative domain convention used for the exit waves.

The third row uses a difference-based magnetic contrast from the ideal FTH reconstructions: `FTH(CR - CL)`. It plots the amplitude, real part, and imaginary part of the positive-minus-negative domain contrast. This is intentionally separate from the multiplicative ratio contrast in the second row.

Phase curves are unwrapped along the energy axis and shifted by a constant multiple of `2*pi` so branch-cut jumps do not appear as physical spectral features. For ratio-domain contrasts, the phase is unwrapped before constructing the complex logarithm, so the plotted phase is continuous along the sweep.


In [ ]:
%matplotlib inline

if energy_results:
    crop = sample_oh_slices
    mz_crop = magnetic_pattern_2d[crop]
    positive_domain = mz_crop > xmcd_domain_threshold
    negative_domain = mz_crop < -xmcd_domain_threshold

    if not np.any(positive_domain) or not np.any(negative_domain):
        raise ValueError(
            "Domain masks are empty. Lower xmcd_domain_threshold or check the magnetic pattern."
        )

    # Create circular aperture mask for real-space OH crop
    oh_index = aperture_types.index("OH")
    oh_radius_m = aperture_radii[oh_index]
    oh_radius_px = oh_radius_m / real_space_pixel_size
    
    # Center of crop in cropped coordinates
    crop_h, crop_w = mz_crop.shape
    cy_crop, cx_crop = crop_h // 2, crop_w // 2
    yy, xx = np.ogrid[:crop_h, :crop_w]
    oh_circular_mask = (yy - cy_crop)**2 + (xx - cx_crop)**2 <= oh_radius_px**2
    
    # Apply circular constraint to domain masks
    positive_domain = positive_domain & oh_circular_mask
    negative_domain = negative_domain & oh_circular_mask

    energies = np.array([result["energy_eV"] for result in energy_results])
    positive_ratio_mean = []
    negative_ratio_mean = []
    fth_positive_ratio_mean = []
    fth_negative_ratio_mean = []
    fth_positive_diff_mean = []
    fth_negative_diff_mean = []
    detected_fth_positive_diff_mean = []
    detected_fth_negative_diff_mean = []
    eps = 1e-30

    for result in energy_results:
        h = result["holograms"]

        cr_crop = frame_average(h.exit_waves["CR"])[crop]
        cl_crop = frame_average(h.exit_waves["CL"])[crop]
        ratio_crop = (cr_crop + eps) / (cl_crop + eps)
        positive_ratio_mean.append(np.nanmean(ratio_crop[positive_domain]))
        negative_ratio_mean.append(np.nanmean(ratio_crop[negative_domain]))

        fth_crop = result["recon_oh_slices"]
        reconstruction_pixel_size = float(result["detector_config"].detector_layout.real_space_resolution)
        
        fth_cr = frame_average(h.reconstructions["ideal"]["CR"])[fth_crop]
        fth_cl = frame_average(h.reconstructions["ideal"]["CL"])[fth_crop]
        fth_diff = frame_average(h.reconstructions["ideal"]["diff"])[fth_crop]
        
        # Create circular aperture mask for FTH-space OH crop
        fth_oh_radius_px = oh_radius_m / reconstruction_pixel_size
        fth_crop_h, fth_crop_w = fth_cr.shape
        fth_cy_crop, fth_cx_crop = fth_crop_h // 2, fth_crop_w // 2
        fth_yy, fth_xx = np.ogrid[:fth_crop_h, :fth_crop_w]
        fth_oh_circular_mask = (fth_yy - fth_cy_crop)**2 + (fth_xx - fth_cx_crop)**2 <= fth_oh_radius_px**2
        
        # Resample domain masks and apply circular FTH mask
        positive_fth_domain = resample_nearest_to_shape(positive_domain, fth_cr.shape).astype(bool)
        negative_fth_domain = resample_nearest_to_shape(negative_domain, fth_cr.shape).astype(bool)
        positive_fth_domain = positive_fth_domain & fth_oh_circular_mask
        negative_fth_domain = negative_fth_domain & fth_oh_circular_mask

        fth_ratio_crop = (fth_cr + eps) / (fth_cl + eps)
        fth_positive_ratio_mean.append(np.nanmean(fth_ratio_crop[positive_fth_domain]))
        fth_negative_ratio_mean.append(np.nanmean(fth_ratio_crop[negative_fth_domain]))
        fth_positive_diff_mean.append(np.nanmean(fth_diff[positive_fth_domain]))
        fth_negative_diff_mean.append(np.nanmean(fth_diff[negative_fth_domain]))

        # Detected FTH reconstructions
        detected_fth_diff = frame_average(h.reconstructions["detected"]["diff"])[fth_crop]
        detected_fth_positive_diff_mean.append(np.nanmean(detected_fth_diff[positive_fth_domain]))
        detected_fth_negative_diff_mean.append(np.nanmean(detected_fth_diff[negative_fth_domain]))

    positive_ratio_mean = np.asarray(positive_ratio_mean)
    negative_ratio_mean = np.asarray(negative_ratio_mean)
    fth_positive_ratio_mean = np.asarray(fth_positive_ratio_mean)
    fth_negative_ratio_mean = np.asarray(fth_negative_ratio_mean)
    fth_positive_diff_mean = np.asarray(fth_positive_diff_mean)
    fth_negative_diff_mean = np.asarray(fth_negative_diff_mean)
    detected_fth_positive_diff_mean = np.asarray(detected_fth_positive_diff_mean)
    detected_fth_negative_diff_mean = np.asarray(detected_fth_negative_diff_mean)

    positive_mean = continuous_complex_log_ratio(
        positive_ratio_mean, np.ones_like(positive_ratio_mean)
    )
    negative_mean = continuous_complex_log_ratio(
        negative_ratio_mean, np.ones_like(negative_ratio_mean)
    )
    # Multiplicative domain contrast: compare the complex domain-averaged ratios.
    domain_contrast = continuous_complex_log_ratio(positive_ratio_mean, negative_ratio_mean)
    fth_positive_mean = continuous_complex_log_ratio(
        fth_positive_ratio_mean, np.ones_like(fth_positive_ratio_mean)
    )
    fth_negative_mean = continuous_complex_log_ratio(
        fth_negative_ratio_mean, np.ones_like(fth_negative_ratio_mean)
    )
    fth_domain_contrast = continuous_complex_log_ratio(
        fth_positive_ratio_mean, fth_negative_ratio_mean
    )
    fth_difference_contrast = fth_positive_diff_mean - fth_negative_diff_mean
    detected_fth_difference_contrast = detected_fth_positive_diff_mean - detected_fth_negative_diff_mean

    fig, axes = plt.subplots(4, 3, figsize=(15, 13), sharex=True)

    def plot_offset_xmcd_row(ax, domain_values, contrast_values, title, ylabel, domain_prefix=""):
        positive_values, negative_values = domain_values
        all_values = np.concatenate([
            np.ravel(positive_values),
            np.ravel(negative_values),
            np.ravel(contrast_values),
        ])
        span = np.nanmax(all_values) - np.nanmin(all_values)
        span = float(span if span > 0 else 1.0)
        domain_offset = 0.75 * span
        contrast_offset = -0.75 * span

        ax.plot(energies, positive_values + domain_offset, "o-", color="tab:blue", label=f"{domain_prefix}mz > threshold")
        ax.plot(energies, negative_values + domain_offset, "o-", color="tab:orange", label=f"{domain_prefix}mz < -threshold")
        ax.plot(energies, contrast_values + contrast_offset, "s--", color="tab:green", label="XMCD contrast")
        ax.fill_between(
            energies,
            contrast_offset,
            contrast_values + contrast_offset,
            color="tab:green",
            alpha=0.1,
        )
        ax.axhline(domain_offset, color="0.5", lw=0.8, alpha=0.5)
        ax.axhline(contrast_offset, color="tab:green", lw=0.8, alpha=0.5)
        ax.text(0.01, 0.91, "domain means", transform=ax.transAxes, color="0.35", va="top")
        ax.text(0.01, 0.09, "XMCD contrast vs 0", transform=ax.transAxes, color="tab:green", va="bottom")
        ax.set_title(title)
        ax.set_ylabel(ylabel)

    plot_offset_xmcd_row(
        axes[0, 0],
        (np.real(positive_mean), np.real(negative_mean)),
        np.real(domain_contrast),
        "Exit-wave $A_{\\mathrm{XMCD}}$",
        "offset $A_{\\mathrm{XMCD}}$ display",
    )
    plot_offset_xmcd_row(
        axes[0, 1],
        (np.imag(positive_mean), np.imag(negative_mean)),
        np.imag(domain_contrast),
        "Exit-wave $\\Theta_{\\mathrm{XMCD}}$",
        "offset $\\Theta_{\\mathrm{XMCD}}$ display (rad)",
    )

    plot_offset_xmcd_row(
        axes[1, 0],
        (np.real(fth_positive_mean), np.real(fth_negative_mean)),
        np.real(fth_domain_contrast),
        "Ideal FTH $A_{\\mathrm{XMCD}}$",
        "offset FTH $A_{\\mathrm{XMCD}}$ display",
        domain_prefix="FTH ",
    )
    plot_offset_xmcd_row(
        axes[1, 1],
        (np.imag(fth_positive_mean), np.imag(fth_negative_mean)),
        np.imag(fth_domain_contrast),
        "Ideal FTH $\\Theta_{\\mathrm{XMCD}}$",
        "offset FTH $\\Theta_{\\mathrm{XMCD}}$ display (rad)",
        domain_prefix="FTH ",
    )

    axes[2, 0].plot(
        energies,
        np.abs(fth_difference_contrast),
        "o-",
        color="tab:red",
        label="ideal FTH diff pos - neg",
    )
    axes[2, 1].plot(
        energies,
        np.real(fth_difference_contrast),
        "o-",
        color="tab:red",
        label="ideal FTH diff pos - neg",
    )
    axes[2, 2].plot(
        energies,
        np.imag(fth_difference_contrast),
        "o-",
        color="tab:red",
        label="ideal FTH diff pos - neg",
    )

    axes[3, 0].plot(
        energies,
        np.abs(detected_fth_difference_contrast),
        "o-",
        color="tab:purple",
        label="detected FTH diff pos - neg",
    )
    axes[3, 1].plot(
        energies,
        np.real(detected_fth_difference_contrast),
        "o-",
        color="tab:purple",
        label="detected FTH diff pos - neg",
    )
    axes[3, 2].plot(
        energies,
        np.imag(detected_fth_difference_contrast),
        "o-",
        color="tab:purple",
        label="detected FTH diff pos - neg",
    )

    blank_axes = [axes[0, 2], axes[1, 2]]
    for ax in blank_axes:
        ax.set_axis_off()

    axes[2, 0].set_title("Ideal FTH difference contrast amplitude")
    axes[2, 0].set_ylabel("$|D_+ - D_-|$")
    axes[2, 1].set_title("Ideal FTH difference contrast real part")
    axes[2, 1].set_ylabel("$\\mathrm{Re}[D_+ - D_-]$")
    axes[2, 2].set_title("Ideal FTH difference contrast imaginary part")
    axes[2, 2].set_ylabel("$\\mathrm{Im}[D_+ - D_-]$")

    axes[3, 0].set_title("Detected FTH difference contrast amplitude")
    axes[3, 0].set_ylabel("$|D_+ - D_-|$")
    axes[3, 1].set_title("Detected FTH difference contrast real part")
    axes[3, 1].set_ylabel("$\\mathrm{Re}[D_+ - D_-]$")
    axes[3, 2].set_title("Detected FTH difference contrast imaginary part")
    axes[3, 2].set_ylabel("$\\mathrm{Im}[D_+ - D_-]$")

    for ax in axes.ravel():
        if ax in blank_axes:
            continue
        ax.set_xlabel("photon energy (eV)")
        ax.grid(True, alpha=0.3)
        ax.legend(loc="best")


    print(f"Positive-domain pixels in exit-wave OH crop: {np.count_nonzero(positive_domain)}")
    print(f"Negative-domain pixels in exit-wave OH crop: {np.count_nonzero(negative_domain)}")
else:
    print("No sweep results available yet.")


plt.show()


## 10. Total Intensity Versus Energy

The edge scan should change the transmitted/scattered intensity because the optical constants vary strongly across the Cobalt L3 and L2 edges. The detected curve includes the beamstop, detector response, and noise; the ideal curve is the noiseless `CR + CL` detector intensity before detector noise.



In [ ]:
if energy_results:
    energies = np.array([result["energy_eV"] for result in energy_results])
    ideal_intensity = np.array([result["total_ideal_intensity"] for result in energy_results])
    detected_counts = np.array([result["total_detected_counts"] for result in energy_results])

    fig, ax1 = plt.subplots(figsize=(7, 4))
    ax1.plot(energies, ideal_intensity, "o-", label="ideal CR + CL intensity")
    ax1.set_xlabel("photon energy (eV)")
    ax1.set_ylabel("ideal intensity (arb.)")
    ax1.grid(True, alpha=0.3)

    ax2 = ax1.twinx()
    ax2.plot(energies, detected_counts, "s--", color="tab:orange", label="detected CR + CL counts")
    ax2.set_ylabel("detected counts")

    lines = ax1.get_lines() + ax2.get_lines()
    ax1.legend(lines, [line.get_label() for line in lines], loc="best")
    ax1.set_title("total hologram intensity across the Co L2,3 edge")
else:
    print("No sweep results available yet.")


## 11. Things To Try

- Increase `energy_values_eV` density around 777 to 780 eV for a finer Co L3 scan.
- Add more points around 792 to 795 eV for the Co L2 edge.
- Set `propagate=True` to compare Jones-only and multislice energy dependence.
- Switch `propagator_method="Scalar"` when you want a faster eigenmode-style approximation.
- Increase `detector_shape` once the notebook behavior is clear.

